In [1]:
import os
import sys
import time
import inspect
import contextlib
import pandas as pd
import numpy as np
import random
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import math
from sklearn.model_selection import GroupKFold

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

In [2]:
def seed_everything(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

BASE_SEED = 42
seed_everything(BASE_SEED)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Загрузка датасетов

In [4]:
TRAIN_PATH = "../datasets/train.parquet"
VALID_PATH = "../datasets/valid.parquet"

# Читаем parquet
train_df = pd.read_parquet(TRAIN_PATH)
valid_df = pd.read_parquet(VALID_PATH)

# Фиксируем структуру колонок (по utils.py)
META_COLS = ["seq_ix", "step_in_seq", "need_prediction"]
FEATURE_COLS = train_df.columns[3:35].tolist()   # 32 фичи
TARGET_COLS  = train_df.columns[35:37].tolist()  # 2 таргета

print("train shape:", train_df.shape)
print("valid shape:", valid_df.shape)
print("features:", len(FEATURE_COLS), "targets:", len(TARGET_COLS))
print("target cols:", TARGET_COLS)

train shape: (10721000, 37)
valid shape: (1444000, 37)
features: 32 targets: 2
target cols: ['t0', 't1']


In [5]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10721000 entries, 0 to 10720999
Data columns (total 37 columns):
 #   Column           Dtype  
---  ------           -----  
 0   seq_ix           int64  
 1   step_in_seq      int64  
 2   need_prediction  int8   
 3   p0               float64
 4   p1               float64
 5   p2               float64
 6   p3               float64
 7   p4               float64
 8   p5               float64
 9   p6               float64
 10  p7               float64
 11  p8               float64
 12  p9               float64
 13  p10              float64
 14  p11              float64
 15  v0               float64
 16  v1               float64
 17  v2               float64
 18  v3               float64
 19  v4               float64
 20  v5               float64
 21  v6               float64
 22  v7               float64
 23  v8               float64
 24  v9               float64
 25  v10              float64
 26  v11              float64
 27  dp0              floa

In [6]:
"""
def cast_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["seq_ix"] = df["seq_ix"].astype(np.int32)
    df["step_in_seq"] = df["step_in_seq"].astype(np.int16)
    df["need_prediction"] = df["need_prediction"].astype(bool)
    df[FEATURE_COLS] = df[FEATURE_COLS].astype(np.float32)
    df[TARGET_COLS] = df[TARGET_COLS].astype(np.float32)
    return df

train_df = cast_df(train_df)
valid_df = cast_df(valid_df)"""

'\ndef cast_df(df: pd.DataFrame) -> pd.DataFrame:\n    df = df.copy()\n    df["seq_ix"] = df["seq_ix"].astype(np.int32)\n    df["step_in_seq"] = df["step_in_seq"].astype(np.int16)\n    df["need_prediction"] = df["need_prediction"].astype(bool)\n    df[FEATURE_COLS] = df[FEATURE_COLS].astype(np.float32)\n    df[TARGET_COLS] = df[TARGET_COLS].astype(np.float32)\n    return df\n\ntrain_df = cast_df(train_df)\nvalid_df = cast_df(valid_df)'

In [7]:
max_train_seq_ix = train_df["seq_ix"].max()

valid_df = valid_df.copy()
valid_df["seq_ix"] = valid_df["seq_ix"] + max_train_seq_ix + 1

In [8]:
train_seq_ids = train_df["seq_ix"].unique()
valid_seq_ids = valid_df["seq_ix"].unique()

print("num train seq:", len(train_seq_ids))
print("num valid seq:", len(valid_seq_ids))
print("train seq range:", train_seq_ids.min(), train_seq_ids.max())
print("valid seq range:", valid_seq_ids.min(), valid_seq_ids.max())

num train seq: 10721
num valid seq: 1444
train seq range: 0 10720
valid seq range: 10721 12164


In [9]:
df_all = pd.concat([train_df, valid_df], axis=0, ignore_index=True)

all_seq_ids = df_all["seq_ix"].unique()

print("all shape:", df_all.shape)
print("num all seq:", len(all_seq_ids))
print("all seq range:", all_seq_ids.min(), all_seq_ids.max())

all shape: (12165000, 37)
num all seq: 12165
all seq range: 0 12164


## Подготовка данных к обучению

### Собираем seqs

In [10]:
def build_sequences(df, feature_cols, target_cols):
    """
    Преобразует DataFrame -> набор независимых sequences:
      X_seqs: (N_seq, 1000, 32) float32
      Y_seqs: (N_seq, 1000, 2)  float32
      M_seqs: (N_seq, 1000)     bool
      seq_ids: (N_seq,)         int
    """
    seq_ids = np.sort(df["seq_ix"].unique())
    n_seq = len(seq_ids)
    
    X_seqs = np.empty((n_seq, 1000, len(feature_cols)), dtype=np.float32)
    Y_seqs = np.empty((n_seq, 1000, len(target_cols)), dtype=np.float32)
    M_seqs = np.empty((n_seq, 1000), dtype=bool)

    # Сгруппируем один раз
    grouped = df.groupby("seq_ix", sort=False)
    
    for i, sid in enumerate(seq_ids):
        g = grouped.get_group(sid).sort_values("step_in_seq")
        
        # (1000, 32)
        X_seqs[i] = g[feature_cols].to_numpy(dtype=np.float32, copy=False)
        # (1000, 2)
        Y_seqs[i] = g[target_cols].to_numpy(dtype=np.float32, copy=False)
        # (1000,)
        M_seqs[i] = g["need_prediction"].to_numpy(dtype=bool, copy=False)

    return X_seqs, Y_seqs, M_seqs, seq_ids

In [11]:
AUX_HORIZONS = [1, 2, 3]

# замени на реальные индексы / имена нужных фич
AUX_FEATURE_COLS = [
    # примеры:
     "p0",
     "v0",
     "dp0",
     "dv0"
]

assert len(AUX_FEATURE_COLS) > 0, "Укажи AUX_FEATURE_COLS"
AUX_FEATURE_IDXS = [FEATURE_COLS.index(c) for c in AUX_FEATURE_COLS]

print("AUX_FEATURE_COLS:", AUX_FEATURE_COLS)
print("AUX_FEATURE_IDXS:", AUX_FEATURE_IDXS)
print("AUX_HORIZONS:", AUX_HORIZONS)

AUX_FEATURE_COLS: ['p0', 'v0', 'dp0', 'dv0']
AUX_FEATURE_IDXS: [0, 12, 24, 28]
AUX_HORIZONS: [1, 2, 3]


In [12]:
def build_aux_from_X(X_seqs, aux_feature_idxs, horizons):
    """
    X_seqs: (N, 1000, 32)

    Возвращает:
      Y_aux: (N, 1000, N_aux) float32
      M_aux: (N, 1000, N_aux) bool

    где N_aux = len(aux_feature_idxs) * len(horizons)
    """
    N, T, _ = X_seqs.shape
    n_feat = len(aux_feature_idxs)
    n_aux = n_feat * len(horizons)

    Y_aux = np.zeros((N, T, n_aux), dtype=np.float32)
    M_aux = np.zeros((N, T, n_aux), dtype=bool)

    col_offset = 0
    for h in horizons:
        # будущие значения выбранных фич
        # для t in [0, T-h-1] target существует
        Y_aux[:, :T-h, col_offset:col_offset+n_feat] = X_seqs[:, h:, aux_feature_idxs]
        M_aux[:, :T-h, col_offset:col_offset+n_feat] = True

        col_offset += n_feat

    return Y_aux, M_aux

In [13]:
X_all, Y_all, M_all, seq_ids_all = build_sequences(df_all, FEATURE_COLS, TARGET_COLS)

print("X_all:", X_all.shape, X_all.dtype)
print("Y_all:", Y_all.shape, Y_all.dtype)
print("M_all:", M_all.shape, M_all.dtype)
print("seq_ids_all:", seq_ids_all.shape, seq_ids_all[:5], seq_ids_all[-5:])

Y_aux_all, M_aux_all = build_aux_from_X(X_all, AUX_FEATURE_IDXS, AUX_HORIZONS)

print("Y_aux_all:", Y_aux_all.shape, Y_aux_all.dtype)
print("M_aux_all:", M_aux_all.shape, M_aux_all.dtype)

X_all: (12165, 1000, 32) float32
Y_all: (12165, 1000, 2) float32
M_all: (12165, 1000) bool
seq_ids_all: (12165,) [0 1 2 3 4] [12160 12161 12162 12163 12164]
Y_aux_all: (12165, 1000, 12) float32
M_aux_all: (12165, 1000, 12) bool


In [14]:
gkf = GroupKFold(n_splits=5)

dummy_X = np.zeros((len(seq_ids_all), 1), dtype=np.float32)
fold_splits = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(dummy_X, groups=seq_ids_all), start=1):
    fold_splits.append((tr_idx, va_idx))
    print(f"Fold {fold}: train={len(tr_idx)}, valid={len(va_idx)}")

Fold 1: train=9732, valid=2433
Fold 2: train=9732, valid=2433
Fold 3: train=9732, valid=2433
Fold 4: train=9732, valid=2433
Fold 5: train=9732, valid=2433


### Собираем Dataloaders

In [15]:
def slice_fold_data(X, Y, M, Y_aux, M_aux, tr_idx, va_idx):
    train_pack = {
        "X": X[tr_idx],
        "Y": Y[tr_idx],
        "M": M[tr_idx],
        "Y_aux": Y_aux[tr_idx],
        "M_aux": M_aux[tr_idx],
    }
    valid_pack = {
        "X": X[va_idx],
        "Y": Y[va_idx],
        "M": M[va_idx],
        "Y_aux": Y_aux[va_idx],
        "M_aux": M_aux[va_idx],
    }
    return train_pack, valid_pack

In [16]:
class FullSequenceDataset(Dataset):
    def __init__(self, X_seqs, Y_seqs, M_seqs, Y_aux_seqs, M_aux_seqs):
        self.X = X_seqs
        self.Y = Y_seqs
        self.M = M_seqs
        self.Y_aux = Y_aux_seqs
        self.M_aux = M_aux_seqs

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.Y[idx],
            self.M[idx],
            self.Y_aux[idx],
            self.M_aux[idx],
        )

class FullSequenceCollator:
    def __call__(self, batch):
        X_list, Y_list, M_list, Y_aux_list, M_aux_list = zip(*batch)

        xb = torch.tensor(np.stack(X_list), dtype=torch.float32)
        yb = torch.tensor(np.stack(Y_list), dtype=torch.float32)
        mb = torch.tensor(np.stack(M_list), dtype=torch.bool)
        y_aux_b = torch.tensor(np.stack(Y_aux_list), dtype=torch.float32)
        m_aux_b = torch.tensor(np.stack(M_aux_list), dtype=torch.bool)

        return xb, yb, mb, y_aux_b, m_aux_b


In [17]:
def make_fullseq_loaders(train_pack, valid_pack, batch_size=32, seed=42):
    collator = FullSequenceCollator()

    train_ds = FullSequenceDataset(
        train_pack["X"], train_pack["Y"], train_pack["M"],
        train_pack["Y_aux"], train_pack["M_aux"]
    )
    valid_ds = FullSequenceDataset(
        valid_pack["X"], valid_pack["Y"], valid_pack["M"],
        valid_pack["Y_aux"], valid_pack["M_aux"]
    )

    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=g,
        collate_fn=collator,
        drop_last=True,
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collator,
        drop_last=False,
    )

    return train_loader, valid_loader

## Обучение

In [18]:
# -------------------------
# Causal TCN building blocks (как у тебя)
# -------------------------
class CausalChomp1d(nn.Module):
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size]


class TCNResidualBlock(nn.Module):
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        kernel_size: int = 3,
        dilation: int = 1,
        dropout: float = 0.1,
        use_norm: bool = True,
    ):
        super().__init__()
        pad = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation)
        self.chomp1 = CausalChomp1d(pad)
        self.act1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation)
        self.chomp2 = CausalChomp1d(pad)
        self.act2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)

        self.downsample = None
        if in_ch != out_ch:
            self.downsample = nn.Conv1d(in_ch, out_ch, kernel_size=1)

        self.norm = nn.GroupNorm(num_groups=1, num_channels=out_ch) if use_norm else nn.Identity()

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.act1(out)
        out = self.drop1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.act2(out)
        out = self.drop2(out)

        if self.downsample is not None:
            residual = self.downsample(residual)

        out = out + residual
        out = self.norm(out)
        return out


def make_tcn_backbone(
    input_dim: int,
    channels=(256, 128, 64),
    kernel_size: int = 3,
    dropout: float = 0.1,
    use_norm: bool = True,
):
    blocks = []
    in_ch = input_dim
    for i, out_ch in enumerate(channels):
        d = 2 ** i  # 1,2,4,...
        blocks.append(
            TCNResidualBlock(
                in_ch=in_ch,
                out_ch=out_ch,
                kernel_size=kernel_size,
                dilation=d,
                dropout=dropout,
                use_norm=use_norm,
            )
        )
        in_ch = out_ch
    return nn.Sequential(*blocks), channels[-1]


# -------------------------
# Variant 2: two independent models from input
# -------------------------
class TwoIndependentTCNRegressor(nn.Module):
    """
    Полностью раздельные ветки для t0 и t1:
      input:  (B, 100, F)
      output: (B, 2)

    Ничего в остальном пайплайне менять не нужно.
    """
    def __init__(
        self,
        input_dim: int,
        channels0=(256, 128, 64),
        channels1=(256, 128, 64),
        kernel_size: int = 3,
        dropout: float = 0.1,
        use_norm: bool = True,
        head_dim_0: int = 64,
        head_dim_1: int = 128,
        head_dropout: float = 0.1,
        clip_out=None,  # оставь None: клип у тебя в лоссе через tanh
    ):
        super().__init__()
        self.clip_out = clip_out

        # Ветка t0
        self.tcn0, last0 = make_tcn_backbone(
            input_dim=input_dim, channels=channels0, kernel_size=kernel_size, dropout=dropout, use_norm=use_norm
        )
        self.head0 = nn.Sequential(
            nn.Linear(last0, head_dim_0),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_dim_0, 1),
        )

        # Ветка t1
        self.tcn1, last1 = make_tcn_backbone(
            input_dim=input_dim, channels=channels1, kernel_size=kernel_size, dropout=dropout, use_norm=use_norm
        )
        self.head1 = nn.Sequential(
            nn.Linear(last1, head_dim_1),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_dim_1, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,T,F) -> Conv1d wants (B,F,T)
        x_c = x.transpose(1, 2)  # (B,F,100)

        z0 = self.tcn0(x_c)      # (B,C0,T)
        z1 = self.tcn1(x_c)      # (B,C1,T)

        last0 = z0[:, :, -1]     # (B,C0)
        last1 = z1[:, :, -1]     # (B,C1)

        y0 = self.head0(last0)   # (B,1)
        y1 = self.head1(last1)   # (B,1)

        out = torch.cat([y0, y1], dim=1)  # (B,2)

        if self.clip_out is not None:
            out = self.clip_out * torch.tanh(out / self.clip_out)

        return out

In [19]:
class SharedTCNMultiTaskRegressor(nn.Module):
    """
    Один shared TCN backbone + 5 голов:
      - t0 main head -> (B,1)
      - t1 main head -> (B,1)
      - aux +1 head  -> (B, aux_dim_per_horizon)
      - aux +2 head  -> (B, aux_dim_per_horizon)
      - aux +3 head  -> (B, aux_dim_per_horizon)

    input:
      x: (B, T, F)

    output:
      main_out: (B, 2)
      aux_dict: {
          1: (B, aux_dim_per_horizon),
          2: (B, aux_dim_per_horizon),
          3: (B, aux_dim_per_horizon),
      }
    """
    def __init__(
        self,
        input_dim: int,
        aux_dim_per_horizon: int,
        aux_horizons=(1, 2, 3),
        channels=(256, 128, 64),
        kernel_size: int = 3,
        dropout: float = 0.1,
        use_norm: bool = True,
        head_dim_main: int = 128,
        head_dim_aux: int = 128,
        head_dropout: float = 0.1,
        clip_out=None,
    ):
        super().__init__()
        self.clip_out = clip_out
        self.aux_horizons = tuple(aux_horizons)
        self.aux_dim_per_horizon = int(aux_dim_per_horizon)

        self.tcn, last_dim = make_tcn_backbone(
            input_dim=input_dim,
            channels=channels,
            kernel_size=kernel_size,
            dropout=dropout,
            use_norm=use_norm,
        )

        # main heads
        self.head_t0 = nn.Sequential(
            nn.Linear(last_dim, head_dim_main),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_dim_main, 1),
        )

        self.head_t1 = nn.Sequential(
            nn.Linear(last_dim, head_dim_main),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_dim_main, 1),
        )

        # aux heads by horizon
        self.aux_heads = nn.ModuleDict({
            str(h): nn.Sequential(
                nn.Linear(last_dim, head_dim_aux),
                nn.GELU(),
                nn.Dropout(head_dropout),
                nn.Linear(head_dim_aux, self.aux_dim_per_horizon),
            )
            for h in self.aux_horizons
        })

    def forward(self, x: torch.Tensor):
        # x: (B,T,F) -> (B,F,T)
        x_c = x.transpose(1, 2)

        z = self.tcn(x_c)         # (B,C,T)
        last = z[:, :, -1]        # (B,C)

        y0 = self.head_t0(last)   # (B,1)
        y1 = self.head_t1(last)   # (B,1)
        main_out = torch.cat([y0, y1], dim=1)  # (B,2)

        if self.clip_out is not None:
            main_out = self.clip_out * torch.tanh(main_out / self.clip_out)

        aux_dict = {
            h: self.aux_heads[str(h)](last) for h in self.aux_horizons
        }

        return main_out, aux_dict

In [20]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 256):
        super().__init__()
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)   # (T, D)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (T,1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, T, D)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T, :]


class ConvDilatedBlock(nn.Module):
    def __init__(self, ch: int, kernel_size: int = 3, dilation: int = 1, dropout: float = 0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(ch, ch, kernel_size=kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(ch, ch, kernel_size=kernel_size, padding=pad, dilation=dilation)

        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.norm = nn.GroupNorm(num_groups=1, num_channels=ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        r = x
        x = self.drop(self.act(self.conv1(x)))
        x = self.drop(self.act(self.conv2(x)))
        x = self.norm(x + r)
        return x


class ConvTransformerMultiTaskRegressor(nn.Module):
    """
    input:  (B, T, F)

    output:
      main_out: (B, 2)
      aux_dict: {
          1: (B, aux_dim_per_horizon),
          2: (B, aux_dim_per_horizon),
          3: (B, aux_dim_per_horizon),
      }
    """
    def __init__(
        self,
        input_dim: int,
        aux_dim_per_horizon: int,
        aux_horizons=(1, 2, 3),
        window: int = 100,
        embed_dim: int = 128,
        conv_blocks: int = 2,
        conv_kernel: int = 3,
        conv_dropout: float = 0.1,
        attn_layers: int = 1,
        attn_heads: int = 4,
        ffn_mult: int = 4,
        attn_dropout: float = 0.1,
        head_dim_main: int = 64,
        head_dim_aux: int = 64,
        head_dropout: float = 0.1,
        clip_out=None,
        pooling: str = "last",   # "last" or "mean"
    ):
        super().__init__()
        assert embed_dim % attn_heads == 0

        self.input_dim = input_dim
        self.aux_dim_per_horizon = int(aux_dim_per_horizon)
        self.aux_horizons = tuple(aux_horizons)
        self.clip_out = clip_out
        self.pooling = pooling

        self.input_proj = nn.Linear(input_dim, embed_dim)
        self.conv_in_norm = nn.LayerNorm(embed_dim)

        self.conv = nn.Sequential(*[
            ConvDilatedBlock(
                ch=embed_dim,
                kernel_size=conv_kernel,
                dilation=(2 ** i),
                dropout=conv_dropout,
            )
            for i in range(conv_blocks)
        ])

        self.pos = SinusoidalPositionalEncoding(embed_dim, max_len=window)

        if attn_layers > 0:
            enc_layer = nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=attn_heads,
                dim_feedforward=embed_dim * ffn_mult,
                dropout=attn_dropout,
                batch_first=True,
                activation="gelu",
                norm_first=True,
            )
            self.transformer = nn.TransformerEncoder(enc_layer, num_layers=attn_layers)
        else:
            self.transformer = None

        self.final_norm = nn.LayerNorm(embed_dim)

        # main heads
        self.head_t0 = nn.Sequential(
            nn.Linear(embed_dim, head_dim_main),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_dim_main, 1),
        )

        self.head_t1 = nn.Sequential(
            nn.Linear(embed_dim, head_dim_main),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_dim_main, 1),
        )

        # aux heads by horizon
        self.aux_heads = nn.ModuleDict({
            str(h): nn.Sequential(
                nn.Linear(embed_dim, head_dim_aux),
                nn.GELU(),
                nn.Dropout(head_dropout),
                nn.Linear(head_dim_aux, self.aux_dim_per_horizon),
            )
            for h in self.aux_horizons
        })

    def _pool_hidden(self, z: torch.Tensor) -> torch.Tensor:
        # z: (B, T, E)
        if self.pooling == "last":
            h = z[:, -1, :]
        elif self.pooling == "mean":
            h = z.mean(dim=1)
        else:
            raise ValueError(f"Unknown pooling: {self.pooling}")
        return self.final_norm(h)

    def forward(self, x: torch.Tensor):
        # x: (B, T, F)
        x = self.input_proj(x)         # (B, T, E)
        x = self.conv_in_norm(x)

        z = x.transpose(1, 2)          # (B, E, T)
        z = self.conv(z)               # (B, E, T)
        z = z.transpose(1, 2)          # (B, T, E)

        z = self.pos(z)                # (B, T, E)

        if self.transformer is not None:
            z = self.transformer(z)    # (B, T, E)

        h = self._pool_hidden(z)       # (B, E)

        y0 = self.head_t0(h)           # (B, 1)
        y1 = self.head_t1(h)           # (B, 1)
        main_out = torch.cat([y0, y1], dim=1)   # (B, 2)

        if self.clip_out is not None:
            main_out = self.clip_out * torch.tanh(main_out / self.clip_out)

        aux_dict = {
            h_: self.aux_heads[str(h_)](h) for h_ in self.aux_horizons
        }

        return main_out, aux_dict

In [21]:
class GRUSeq2SeqMultiTaskRegressor(nn.Module):
    """
    Full-sequence causal GRU multi-task model.

    input:
      x: (B, T, F)

    output:
      main_out: (B, T, 2)
      aux_dict: {
          1: (B, T, aux_dim_per_horizon),
          2: (B, T, aux_dim_per_horizon),
          3: (B, T, aux_dim_per_horizon),
      }
    """
    def __init__(
        self,
        input_dim: int,
        aux_dim_per_horizon: int,
        aux_horizons=(1, 2, 3),
        hidden_dim: int = 128,
        num_layers: int = 3,
        dropout: float = 0.1,
        clip_out=None,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.aux_dim_per_horizon = int(aux_dim_per_horizon)
        self.aux_horizons = tuple(aux_horizons)
        self.hidden_dim = int(hidden_dim)
        self.num_layers = int(num_layers)
        self.clip_out = clip_out

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )

        # linear head per target, applied at every timestep
        self.head_t0 = nn.Linear(hidden_dim, 1)
        self.head_t1 = nn.Linear(hidden_dim, 1)

        # linear aux head per horizon, applied at every timestep
        self.aux_heads = nn.ModuleDict({
            str(h): nn.Linear(hidden_dim, self.aux_dim_per_horizon)
            for h in self.aux_horizons
        })

    def forward(self, x: torch.Tensor):
        # x: (B, T, F)
        h_seq, _ = self.gru(x)   # (B, T, H)

        y0 = self.head_t0(h_seq)   # (B, T, 1)
        y1 = self.head_t1(h_seq)   # (B, T, 1)
        main_out = torch.cat([y0, y1], dim=-1)   # (B, T, 2)

        if self.clip_out is not None:
            main_out = self.clip_out * torch.tanh(main_out / self.clip_out)

        aux_dict = {
            h_: self.aux_heads[str(h_)](h_seq) for h_ in self.aux_horizons
        }

        return main_out, aux_dict

In [22]:
# ---------------------------
# Loss: Weighted Huber (SmoothL1) over 2 targets
# ---------------------------
def weighted_huber_loss(pred, y, delta=1.0, eps=1e-3):
    """
    pred: (B, 2)
    y:    (B, 2)
    weights w = |y| (clamped), applied elementwise to both targets.
    """
    w = torch.abs(y).clamp_min(eps)  # (B,2)
    hub = F.smooth_l1_loss(pred, y, reduction="none", beta=delta)  # (B,2)
    return (w * hub).mean()


In [23]:
def weighted_huber_loss(
    pred: torch.Tensor,
    y: torch.Tensor,
    delta: float = 1.0,
    eps: float = 1e-3,
    reduction: str = "mean",
) -> torch.Tensor:
    """
    pred: (B, D)
    y:    (B, D)

    weights w = |y| (clamped), applied elementwise.

    reduction:
      - "mean" -> scalar
      - "sum"  -> scalar
      - "none" -> (B, D)
    """
    w = torch.abs(y).clamp_min(eps)   # (B,D)
    hub = F.smooth_l1_loss(pred, y, reduction="none", beta=delta)  # (B,D)
    out = w * hub

    if reduction == "none":
        return out
    elif reduction == "sum":
        return out.sum()
    elif reduction == "mean":
        return out.mean()
    else:
        raise ValueError(f"Unknown reduction: {reduction}")

In [24]:
def hybrid_weighted_huber_corr_loss(
    pred: torch.Tensor,
    y: torch.Tensor,
    delta: float = 1.0,
    eps: float = 1e-8,
    clip: float = 6.0,
    huber_w: float = 0.2,   # <-- маленький вес, чтобы corr доминировал
    corr_w: float = 0.8,    # <-- основной вес
) -> torch.Tensor:
    """
    pred, y: (B, 2)

    - Huber часть: weighted SmoothL1, веса = |y| (как в метрике)
    - Corr часть: 1 - weighted Pearson corr (по каждому таргету), затем среднее по таргетам
    - pred мягко "клипается" в [-clip, clip] через tanh, чтобы соответствовать scorer'у
    """

    # 1) мягкий клип как в метрике (в метрике clamp, тут tanh чтобы градиенты не умирали)
    pred_c = clip * torch.tanh(pred / clip)

    # 2) веса как в scorer'е: |y|, но не даём нулям убить статистики
    w = torch.abs(y).clamp_min(eps)  # (B,2)

    # ---------- Weighted Huber ----------
    hub = F.smooth_l1_loss(pred_c, y, reduction="none", beta=delta)  # (B,2)
    huber_loss = (w * hub).mean()

    # ---------- Weighted Pearson Corr (по batch-оси) ----------
    # sum_w по каждому таргету
    sum_w = w.sum(dim=0).clamp_min(eps)  # (2,)

    mean_y = (w * y).sum(dim=0) / sum_w          # (2,)
    mean_p = (w * pred_c).sum(dim=0) / sum_w     # (2,)

    dev_y = y - mean_y
    dev_p = pred_c - mean_p

    cov = (w * dev_y * dev_p).sum(dim=0) / sum_w           # (2,)
    var_y = (w * dev_y.pow(2)).sum(dim=0) / sum_w          # (2,)
    var_p = (w * dev_p.pow(2)).sum(dim=0) / sum_w          # (2,)

    corr = cov / (torch.sqrt(var_y * var_p).clamp_min(eps))  # (2,)
    corr = torch.clamp(corr, -1.0, 1.0)

    corr_loss = (1.0 - corr).mean()  # хотим corr -> 1, значит loss -> 0

    return huber_w * huber_loss + corr_w * corr_loss

In [25]:
def masked_weighted_npcc_loss_seq2seq(
    pred: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    eps: float = 1e-8,
    clip: float = 6.0,
) -> torch.Tensor:
    """
    pred: (B, T, 2)
    y:    (B, T, 2)
    mask: (B, T) bool or float

    Считает weighted NPCC отдельно по 2 таргетам,
    используя только элементы, где mask == 1.
    """
    if mask.dtype != pred.dtype:
        mask = mask.float()

    pred_c = clip * torch.tanh(pred / clip)   # (B,T,2)

    losses = []

    for j in range(y.shape[-1]):
        pred_j = pred_c[..., j]   # (B,T)
        y_j = y[..., j]           # (B,T)

        valid = mask > 0          # (B,T)

        pred_v = pred_j[valid]    # (N,)
        y_v = y_j[valid]          # (N,)

        if pred_v.numel() == 0:
            losses.append(torch.zeros((), device=pred.device, dtype=pred.dtype))
            continue

        w = torch.abs(y_v).clamp_min(eps)
        sum_w = w.sum().clamp_min(eps)

        mean_y = (w * y_v).sum() / sum_w
        mean_p = (w * pred_v).sum() / sum_w

        dev_y = y_v - mean_y
        dev_p = pred_v - mean_p

        cov = (w * dev_y * dev_p).sum() / sum_w
        var_y = (w * dev_y.pow(2)).sum() / sum_w
        var_p = (w * dev_p.pow(2)).sum() / sum_w

        corr = cov / torch.sqrt((var_y * var_p).clamp_min(eps))
        corr = torch.clamp(corr, -1.0, 1.0)

        losses.append(1.0 - corr)

    return torch.stack(losses).mean()

In [26]:
def masked_weighted_wpc_loss_seq2seq(
    pred: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    eps: float = 1e-8,
    clip: float = 6.0,
    t0_weight: float = 0.5,
    t1_weight: float = 0.5,
) -> torch.Tensor:
    """
    pred: (B, T, 2)
    y:    (B, T, 2)
    mask: (B, T) bool or float

    Weighted Pearson / WPC loss по 2 таргетам с отдельными весами.
    loss = t0_weight * (1 - wpc_t0) + t1_weight * (1 - wpc_t1)

    Важно:
    - pred клипается как в scorer
    - считаем только по элементам, где mask == 1
    """
    if mask.dtype != pred.dtype:
        mask = mask.float()

    weight_sum = t0_weight + t1_weight
    if weight_sum <= 0:
        raise ValueError("t0_weight + t1_weight must be > 0")

    t0_weight = t0_weight / weight_sum
    t1_weight = t1_weight / weight_sum

    pred_c = torch.clamp(pred, -clip, clip)   # ближе к scorer, чем tanh

    losses = []

    for j in range(y.shape[-1]):
        pred_j = pred_c[..., j]   # (B,T)
        y_j = y[..., j]           # (B,T)

        valid = mask > 0
        pred_v = pred_j[valid]    # (N,)
        y_v = y_j[valid]          # (N,)

        if pred_v.numel() == 0:
            losses.append(torch.zeros((), device=pred.device, dtype=pred.dtype))
            continue

        w = torch.abs(y_v).clamp_min(eps)
        sum_w = w.sum().clamp_min(eps)

        mean_y = (w * y_v).sum() / sum_w
        mean_p = (w * pred_v).sum() / sum_w

        dev_y = y_v - mean_y
        dev_p = pred_v - mean_p

        cov = (w * dev_y * dev_p).sum() / sum_w
        var_y = (w * dev_y.pow(2)).sum() / sum_w
        var_p = (w * dev_p.pow(2)).sum() / sum_w

        corr = cov / torch.sqrt((var_y * var_p).clamp_min(eps))
        corr = torch.clamp(corr, -1.0, 1.0)

        losses.append(1.0 - corr)

    loss_t0, loss_t1 = losses
    return t0_weight * loss_t0 + t1_weight * loss_t1

In [27]:
def masked_weighted_huber_loss_seq2seq(
    pred: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    delta: float = 1.0,
    eps: float = 1e-3,
) -> torch.Tensor:
    """
    pred: (B, T, D)
    y:    (B, T, D)
    mask: (B, T, D) bool or float

    Weighted Huber only on valid elements.
    """
    if mask.dtype != pred.dtype:
        mask = mask.float()

    w = torch.abs(y).clamp_min(eps)  # (B,T,D)
    hub = F.smooth_l1_loss(pred, y, reduction="none", beta=delta)  # (B,T,D)

    loss_raw = w * hub * mask
    denom = mask.sum().clamp_min(1.0)

    return loss_raw.sum() / denom

In [28]:
# aux loss: unreduced, mask накладываем внутри train_model
def aux_loss_fn(pred, y, delta=1.0, eps=1e-3):
    return weighted_huber_loss(
        pred,
        y,
        delta=delta,
        eps=eps,
        reduction="none",
    )

In [29]:
def split_aux_by_horizon_seq2seq(
    y_aux: torch.Tensor,
    m_aux: torch.Tensor,
    aux_horizons,
    aux_dim_per_horizon: int,
):
    """
    y_aux: (B, T, D_total)
    m_aux: (B, T, D_total)
    """
    aux_targets = {}
    aux_masks = {}

    for i, h in enumerate(aux_horizons):
        start = i * aux_dim_per_horizon
        end = (i + 1) * aux_dim_per_horizon

        aux_targets[h] = y_aux[:, :, start:end]   # (B,T,D_h)
        aux_masks[h] = m_aux[:, :, start:end]     # (B,T,D_h)

    return aux_targets, aux_masks

In [30]:
def build_main_loss_mask(
    m_main: torch.Tensor,
    warmup: int = 100,
) -> torch.Tensor:
    """
    m_main: (B, T) bool
    Возвращает mask (B, T), где:
      - берем только valid main positions
      - исключаем первые warmup шагов
    """
    B, T = m_main.shape
    step_mask = torch.arange(T, device=m_main.device).unsqueeze(0) >= warmup
    return m_main & step_mask

In [31]:
def masked_weighted_huber_loss_per_channel_seq2seq(
    pred: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    delta: float = 1.0,
    eps: float = 1e-3,
) -> torch.Tensor:
    """
    pred: (B, T, D)
    y:    (B, T, D)
    mask: (B, T, D) bool or float

    Возвращает loss по каждому aux-каналу отдельно: (D,)
    """
    if mask.dtype != pred.dtype:
        mask = mask.float()

    w = torch.abs(y).clamp_min(eps)                          # (B,T,D)
    hub = F.smooth_l1_loss(pred, y, reduction="none", beta=delta)  # (B,T,D)

    raw = w * hub * mask                                     # (B,T,D)

    denom = mask.sum(dim=(0, 1)).clamp_min(1.0)              # (D,)
    loss_per_channel = raw.sum(dim=(0, 1)) / denom           # (D,)

    return loss_per_channel

In [32]:
# ---------------------------
# Init + one training step sanity check
# ---------------------------



"""


model = TwoIndependentTCNRegressor(
    input_dim=len(FEATURE_COLS),
    channels0=(64, 64, 32),
    channels1=(128, 128, 64),      # можно сделать для t1 мощнее: (256, 256, 128, 64) но дороже
    kernel_size=3,
    dropout=0.1,
    use_norm=True,
    head_dim_0=32,
    head_dim_1=64,
    head_dropout=0.1,
    clip_out=None,
).to(device)





optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

"""

'\n\n\nmodel = TwoIndependentTCNRegressor(\n    input_dim=len(FEATURE_COLS),\n    channels0=(64, 64, 32),\n    channels1=(128, 128, 64),      # можно сделать для t1 мощнее: (256, 256, 128, 64) но дороже\n    kernel_size=3,\n    dropout=0.1,\n    use_norm=True,\n    head_dim_0=32,\n    head_dim_1=64,\n    head_dropout=0.1,\n    clip_out=None,\n).to(device)\n\n\n\n\n\noptimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)\n\n'

In [33]:
AUX_DIM_TOTAL = Y_aux_all.shape[-1]
AUX_DIM_PER_HORIZON = AUX_DIM_TOTAL // len(AUX_HORIZONS)

print("AUX_DIM_TOTAL:", AUX_DIM_TOTAL)
print("AUX_DIM_PER_HORIZON:", AUX_DIM_PER_HORIZON)

assert AUX_DIM_TOTAL % len(AUX_HORIZONS) == 0

AUX_DIM_TOTAL: 12
AUX_DIM_PER_HORIZON: 4


In [34]:
model = GRUSeq2SeqMultiTaskRegressor(
    input_dim=len(FEATURE_COLS),
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_horizons=AUX_HORIZONS,
    hidden_dim=128,
    num_layers=3,
    dropout=0.1,
    clip_out=None,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4,
)

In [35]:
"""
# берём батч окон из нашего train_loader
xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)

model.train()
pred = model(xb)
print("pred shape:", pred.shape, pred.dtype)  # (B,2)

loss = weighted_huber_loss(pred, yb, delta=1.0)
print("loss:", float(loss))

optimizer.zero_grad(set_to_none=True)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # небольшая страховка от взрывов
optimizer.step()

print("one optimizer step done")"""

'\n# берём батч окон из нашего train_loader\nxb, yb = next(iter(train_loader))\nxb, yb = xb.to(device), yb.to(device)\n\nmodel.train()\npred = model(xb)\nprint("pred shape:", pred.shape, pred.dtype)  # (B,2)\n\nloss = weighted_huber_loss(pred, yb, delta=1.0)\nprint("loss:", float(loss))\n\noptimizer.zero_grad(set_to_none=True)\nloss.backward()\ntorch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # небольшая страховка от взрывов\noptimizer.step()\n\nprint("one optimizer step done")'

In [36]:
# ---------------------------
# Weighted Pearson (exactly like utils.py logic)
# ---------------------------
def weighted_pearson_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # clip predictions
    y_pred = np.clip(y_pred, -6.0, 6.0)

    weights = np.abs(y_true)
    weights = np.maximum(weights, 1e-8)

    sum_w = np.sum(weights)
    if sum_w == 0:
        return 0.0

    mean_true = np.sum(y_true * weights) / sum_w
    mean_pred = np.sum(y_pred * weights) / sum_w

    dev_true = y_true - mean_true
    dev_pred = y_pred - mean_pred

    cov = np.sum(weights * dev_true * dev_pred) / sum_w
    var_true = np.sum(weights * dev_true**2) / sum_w
    var_pred = np.sum(weights * dev_pred**2) / sum_w

    if var_true <= 0 or var_pred <= 0:
        return 0.0

    return float(cov / (np.sqrt(var_true) * np.sqrt(var_pred)))


def weighted_pearson_2targets(y_true_2: np.ndarray, y_pred_2: np.ndarray) -> dict:
    # y_true_2, y_pred_2: (N, 2)
    t0 = weighted_pearson_np(y_true_2[:, 0], y_pred_2[:, 0])
    t1 = weighted_pearson_np(y_true_2[:, 1], y_pred_2[:, 1])
    return {"t0": t0, "t1": t1, "weighted_pearson": 0.5 * (t0 + t1)}

# ---------------------------
# Regression metrics for 2 targets
# ---------------------------
def regression_metrics_2targets(y_true_2: np.ndarray, y_pred_2: np.ndarray) -> dict:
    """
    y_true_2, y_pred_2: (N, 2)

    Возвращает только метрики по каждому таргету:
      t0_mse, t0_rmse, t0_mae, t0_r2
      t1_mse, t1_rmse, t1_mae, t1_r2
    """
    y_true_2 = np.asarray(y_true_2, dtype=np.float64)
    y_pred_2 = np.asarray(y_pred_2, dtype=np.float64)

    out = {}

    for i, name in enumerate(["t0", "t1"]):
        y = y_true_2[:, i]
        p = y_pred_2[:, i]

        err = p - y
        mse = float(np.mean(err ** 2))
        rmse = float(np.sqrt(mse))
        mae = float(np.mean(np.abs(err)))

        y_mean = float(np.mean(y))
        ss_res = float(np.sum((y - p) ** 2))
        ss_tot = float(np.sum((y - y_mean) ** 2))
        r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 1e-12 else float("nan")

        out[f"{name}_mse"] = mse
        out[f"{name}_rmse"] = rmse
        out[f"{name}_mae"] = mae
        out[f"{name}_r2"] = r2

    return out

In [37]:
# ---------------------------
# Full valid evaluation: run ALL scored windows for ALL valid sequences
# (window-based inference identical to prod logic)
# ---------------------------
@torch.no_grad()
def evaluate_full_valid(
    model,
    X_valid: np.ndarray,  # (N_seq, 1000, 32) float32
    Y_valid: np.ndarray,  # (N_seq, 1000, 2)  float32
    M_valid: np.ndarray,  # (N_seq, 1000)     bool
    device,
    window=100,
    eval_batch_size=512,
    loss_fn=None,                 # <-- NEW
    loss_kwargs=None,             # <-- NEW (e.g., {"delta": 1.0})
):
    model.eval()
    if loss_kwargs is None:
        loss_kwargs = {}

    preds_all = []
    y_all = []

    total_loss = 0.0
    total_batches = 0

    X_buf, Y_buf = [], []

    n_seq = X_valid.shape[0]

    pbar = tqdm(range(n_seq), desc="Valid inference (full)", leave=False)
    for s in pbar:
        # scored t indices in this seq
        ts = np.where(M_valid[s])[0]
        ts = ts[ts >= (window - 1)]  # safety

        X_seq = X_valid[s]
        Y_seq = Y_valid[s]

        for t in ts:
            start = t - (window - 1)
            end = t + 1
            X_buf.append(X_seq[start:end, :])   # (window,32)
            Y_buf.append(Y_seq[t, :])           # (2,)

            if len(X_buf) >= eval_batch_size:
                xb = torch.tensor(np.stack(X_buf), dtype=torch.float32, device=device)
                yb = torch.tensor(np.stack(Y_buf), dtype=torch.float32, device=device)

                pred = model(xb)  # (B,2)

                # loss (generic)
                if loss_fn is not None:
                    loss = loss_fn(pred, yb, **loss_kwargs)
                    total_loss += float(loss.detach().item())
                    total_batches += 1

                # store for pearson (cpu numpy)
                preds_all.append(pred.detach().cpu().numpy())
                y_all.append(yb.detach().cpu().numpy())

                X_buf, Y_buf = [], []

    # flush остаток
    if len(X_buf) > 0:
        xb = torch.tensor(np.stack(X_buf), dtype=torch.float32, device=device)
        yb = torch.tensor(np.stack(Y_buf), dtype=torch.float32, device=device)
        pred = model(xb)

        if loss_fn is not None:
            loss = loss_fn(pred, yb, **loss_kwargs)
            total_loss += float(loss.detach().item())
            total_batches += 1

        preds_all.append(pred.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    preds_all = np.concatenate(preds_all, axis=0)  # (N,2)
    y_all = np.concatenate(y_all, axis=0)          # (N,2)

    pearson = weighted_pearson_2targets(y_all, preds_all)
    valid_loss = total_loss / max(total_batches, 1) if loss_fn is not None else float("nan")

    return valid_loss, pearson


@torch.no_grad()
def evaluate_fast_valid(
    model,
    valid_loader,
    device,
    loss_fn,
    loss_kwargs=None,         # <-- NEW
):
    """
    Быстрая валидация: берём окна из valid_loader и считаем:
      - valid_loss (любой loss_fn)
      - valid_weighted_pearson (weighted pearson по всем собранным предиктам)
    """
    model.eval()
    if loss_kwargs is None:
        loss_kwargs = {}

    preds_all = []
    y_all = []

    loss_sum = 0.0
    iters = 0

    with tqdm(total=len(valid_loader), desc="Valid (fast)", unit="batch", leave=False) as pbar:
        for xb, yb in valid_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            pred = model(xb)  # (B,2)
            loss = loss_fn(pred, yb, **loss_kwargs)

            loss_sum += float(loss.detach().item())
            iters += 1

            preds_all.append(pred.detach().cpu().numpy())
            y_all.append(yb.detach().cpu().numpy())

            pbar.update(1)

    preds_all = np.concatenate(preds_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)

    pearson = weighted_pearson_2targets(y_all, preds_all)
    valid_loss = loss_sum / max(iters, 1)
    return valid_loss, pearson

# ---------------------------
# FULL valid
# ---------------------------
@torch.no_grad()
def evaluate_full_valid(
    model,
    X_valid: np.ndarray,  # (N_seq, T, F)
    Y_valid: np.ndarray,  # (N_seq, T, 2)
    M_valid: np.ndarray,  # (N_seq, T) bool
    device,
    window=100,
    eval_batch_size=512,
    loss_fn=None,
    loss_kwargs=None,
):
    model.eval()
    if loss_kwargs is None:
        loss_kwargs = {}

    preds_all = []
    y_all = []

    total_loss = 0.0
    total_batches = 0

    X_buf, Y_buf = [], []

    n_seq = X_valid.shape[0]

    pbar = tqdm(range(n_seq), desc="Valid inference (full)", leave=False)
    for s in pbar:
        ts = np.where(M_valid[s])[0]
        ts = ts[ts >= (window - 1)]

        X_seq = X_valid[s]
        Y_seq = Y_valid[s]

        for t in ts:
            start = t - (window - 1)
            end = t + 1
            X_buf.append(X_seq[start:end, :])   # (window, F)
            Y_buf.append(Y_seq[t, :])           # (2,)

            if len(X_buf) >= eval_batch_size:
                xb = torch.tensor(np.stack(X_buf), dtype=torch.float32, device=device)
                yb = torch.tensor(np.stack(Y_buf), dtype=torch.float32, device=device)

                pred_main, _ = model(xb)  # (B,2)

                if loss_fn is not None:
                    loss = loss_fn(pred_main, yb, **loss_kwargs)
                    total_loss += float(loss.detach().item())
                    total_batches += 1

                preds_all.append(pred_main.detach().cpu().numpy())
                y_all.append(yb.detach().cpu().numpy())

                X_buf, Y_buf = [], []

    if len(X_buf) > 0:
        xb = torch.tensor(np.stack(X_buf), dtype=torch.float32, device=device)
        yb = torch.tensor(np.stack(Y_buf), dtype=torch.float32, device=device)

        pred_main, _ = model(xb)

        if loss_fn is not None:
            loss = loss_fn(pred_main, yb, **loss_kwargs)
            total_loss += float(loss.detach().item())
            total_batches += 1

        preds_all.append(pred_main.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    preds_all = np.concatenate(preds_all, axis=0)  # (N,2)
    y_all = np.concatenate(y_all, axis=0)          # (N,2)

    pearson = weighted_pearson_2targets(y_all, preds_all)
    reg_metrics = regression_metrics_2targets(y_all, preds_all)

    valid_loss = total_loss / max(total_batches, 1) if loss_fn is not None else float("nan")

    return valid_loss, pearson, reg_metrics


# ---------------------------
# FAST valid
# ---------------------------
@torch.no_grad()
def evaluate_fast_valid(
    model,
    valid_loader,
    device,
    loss_fn,
    loss_kwargs=None,
):
    """
    Быстрая валидация:
      - valid_loss
      - weighted pearson
      - mse/rmse/mae/r2 по каждому таргету
    """
    model.eval()
    if loss_kwargs is None:
        loss_kwargs = {}

    preds_all = []
    y_all = []

    loss_sum = 0.0
    iters = 0

    with tqdm(total=len(valid_loader), desc="Valid (fast)", unit="batch", leave=False) as pbar:
        for xb, yb, _, _ in valid_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            pred_main, _ = model(xb)  # (B,2)
            loss = loss_fn(pred_main, yb, **loss_kwargs)

            loss_sum += float(loss.detach().item())
            iters += 1

            preds_all.append(pred_main.detach().cpu().numpy())
            y_all.append(yb.detach().cpu().numpy())

            pbar.update(1)

    preds_all = np.concatenate(preds_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)

    pearson = weighted_pearson_2targets(y_all, preds_all)
    reg_metrics = regression_metrics_2targets(y_all, preds_all)

    valid_loss = loss_sum / max(iters, 1)
    return valid_loss, pearson, reg_metrics


In [38]:
# ---------------------------
# FULL valid (seq2seq)
# ---------------------------
@torch.no_grad()
def evaluate_full_valid(
    model,
    X_valid: np.ndarray,   # (N_seq, T, F)
    Y_valid: np.ndarray,   # (N_seq, T, 2)
    M_valid: np.ndarray,   # (N_seq, T) bool
    device,
    main_loss_fn,
    main_loss_kwargs=None,
    batch_size=32,
    warmup=100,
):
    model.eval()
    if main_loss_kwargs is None:
        main_loss_kwargs = {}

    preds_all = []
    y_all = []

    loss_sum = 0.0
    iters = 0

    n_seq = X_valid.shape[0]

    with tqdm(total=(n_seq + batch_size - 1) // batch_size, desc="Valid (full)", unit="batch", leave=False) as pbar:
        for start in range(0, n_seq, batch_size):
            end = min(start + batch_size, n_seq)

            xb = torch.tensor(X_valid[start:end], dtype=torch.float32, device=device)   # (B,T,F)
            yb = torch.tensor(Y_valid[start:end], dtype=torch.float32, device=device)   # (B,T,2)
            mb = torch.tensor(M_valid[start:end], dtype=torch.bool, device=device)       # (B,T)

            pred_main, _ = model(xb)   # (B,T,2)

            main_mask = build_main_loss_mask(mb, warmup=warmup)   # (B,T)

            loss = main_loss_fn(pred_main, yb, main_mask, **main_loss_kwargs)
            loss_sum += float(loss.detach().item())
            iters += 1

            valid = main_mask.bool()   # (B,T)

            pred_valid = pred_main[valid]   # (N,2)
            y_valid_t = yb[valid]           # (N,2)

            preds_all.append(pred_valid.detach().cpu().numpy())
            y_all.append(y_valid_t.detach().cpu().numpy())

            pbar.update(1)

    preds_all = np.concatenate(preds_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)

    pearson = weighted_pearson_2targets(y_all, preds_all)
    valid_loss = loss_sum / max(iters, 1)

    return valid_loss, pearson


# ---------------------------
# FAST valid (seq2seq)
# ---------------------------
@torch.no_grad()
def evaluate_fast_valid(
    model,
    valid_loader,
    device,
    main_loss_fn,
    main_loss_kwargs=None,
    warmup=100,
):
    model.eval()
    if main_loss_kwargs is None:
        main_loss_kwargs = {}

    preds_all = []
    y_all = []

    loss_sum = 0.0
    iters = 0

    with tqdm(total=len(valid_loader), desc="Valid (fast)", unit="batch", leave=False) as pbar:
        for xb, yb, mb, y_aux_b, m_aux_b in valid_loader:
            xb = xb.to(device, non_blocking=True)         # (B,T,F)
            yb = yb.to(device, non_blocking=True)         # (B,T,2)
            mb = mb.to(device, non_blocking=True)         # (B,T)

            pred_main, _ = model(xb)                      # (B,T,2)

            main_mask = build_main_loss_mask(mb, warmup=warmup)   # (B,T)

            loss = main_loss_fn(pred_main, yb, main_mask, **main_loss_kwargs)
            loss_sum += float(loss.detach().item())
            iters += 1

            valid = main_mask.bool()

            pred_valid = pred_main[valid]   # (N,2)
            y_valid_t = yb[valid]           # (N,2)

            preds_all.append(pred_valid.detach().cpu().numpy())
            y_all.append(y_valid_t.detach().cpu().numpy())

            pbar.update(1)

    preds_all = np.concatenate(preds_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)

    pearson = weighted_pearson_2targets(y_all, preds_all)
    valid_loss = loss_sum / max(iters, 1)

    return valid_loss, pearson


In [39]:
# ---------------------------
# Silence stdout/stderr during ONNX export
# ---------------------------
@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, "w") as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        try:
            sys.stdout, sys.stderr = devnull, devnull
            yield
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr

def export_onnx_silent(model_cpu, onnx_path, window=100, input_dim=32, opset=18):
    """
    Тихо экспортирует модель в ONNX на CPU.
    Форсит legacy exporter (dynamo=False) если доступно, чтобы меньше зависимостей/логов.
    """
    model_cpu.eval()
    dummy = torch.zeros(1, window, input_dim, dtype=torch.float32)  # CPU

    export_kwargs = dict(
        input_names=["x"],
        output_names=["y"],
        opset_version=opset,
        dynamic_axes={"x": {0: "batch"}, "y": {0: "batch"}},
    )

    sig = inspect.signature(torch.onnx.export)
    if "dynamo" in sig.parameters:
        export_kwargs["dynamo"] = False

    with suppress_output():
        torch.onnx.export(model_cpu, dummy, onnx_path, **export_kwargs)

In [40]:
# ---------------------------
# Train loop (seq2seq)
# ---------------------------
def train_model(
    model_nn,
    train_loader,
    optimizer,
    num_epochs,
    device,
    X_valid, Y_valid, M_valid,

    main_loss_fn,
    main_loss_kwargs=None,

    aux_loss_fn=None,
    aux_loss_kwargs=None,

    aux_horizons=(1, 2, 3),
    aux_dim_per_horizon=4,
    aux_loss_weight=0.05,
    warmup=100,

    valid_loader=None,
    use_full_valid=True,
    full_valid_every=0,

    save_dir_pth="models_pth",
    save_each_epoch=True,
):
    if main_loss_kwargs is None:
        main_loss_kwargs = {}
    if aux_loss_kwargs is None:
        aux_loss_kwargs = {}

    os.makedirs(save_dir_pth, exist_ok=True)

    model_nn = model_nn.to(device)

    for epoch in range(num_epochs):
        start_time = time.time()

        # ---- TRAIN ----
        model_nn.train()
        train_loss_sum = 0.0
        train_iters = 0

        with tqdm(
            total=len(train_loader),
            desc=f"Epoch {epoch+1} - Train",
            unit="batch",
            leave=False,
            dynamic_ncols=True,
        ) as pbar:
            for xb, yb, mb, y_aux_b, m_aux_b in train_loader:
                xb = xb.to(device, non_blocking=True)           # (B,T,F)
                yb = yb.to(device, non_blocking=True)           # (B,T,2)
                mb = mb.to(device, non_blocking=True)           # (B,T)
                y_aux_b = y_aux_b.to(device, non_blocking=True) # (B,T,D_total)
                m_aux_b = m_aux_b.to(device, non_blocking=True) # (B,T,D_total)

                pred_main, pred_aux_dict = model_nn(xb)

                # ---- MAIN LOSS ----
                main_mask = build_main_loss_mask(mb, warmup=warmup)   # (B,T)
                loss_main = main_loss_fn(pred_main, yb, main_mask, **main_loss_kwargs)

                # ---- AUX LOSS ----
                aux_targets, aux_masks = split_aux_by_horizon_seq2seq(
                    y_aux_b,
                    m_aux_b,
                    aux_horizons=aux_horizons,
                    aux_dim_per_horizon=aux_dim_per_horizon,
                )

                aux_loss_parts = []
                for h in aux_horizons:
                    pred_aux_h = pred_aux_dict[h]     # (B,T,D_h)
                    y_aux_h = aux_targets[h]          # (B,T,D_h)
                    m_aux_h = aux_masks[h]            # (B,T,D_h)

                    loss_aux_h = aux_loss_fn(
                        pred_aux_h,
                        y_aux_h,
                        m_aux_h,
                        **aux_loss_kwargs,
                    )
                    aux_loss_parts.append(loss_aux_h)

                if len(aux_loss_parts) > 0:
                    loss_aux = torch.stack(aux_loss_parts).mean()
                else:
                    loss_aux = torch.zeros((), device=device)

                loss = loss_main + aux_loss_weight * loss_aux

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model_nn.parameters(), max_norm=1.0)
                optimizer.step()

                train_loss_sum += float(loss.detach().item())
                train_iters += 1
                pbar.update(1)

        train_loss_out = train_loss_sum / max(train_iters, 1)

        # ---- VALID ----
        do_full = False
        if use_full_valid:
            if full_valid_every == 0:
                do_full = True
            else:
                do_full = (epoch % (full_valid_every + 1) == 0)

        if do_full:
            valid_loss_out, pearson = evaluate_full_valid(
                model_nn,
                X_valid=X_valid,
                Y_valid=Y_valid,
                M_valid=M_valid,
                device=device,
                main_loss_fn=main_loss_fn,
                main_loss_kwargs=main_loss_kwargs,
                batch_size=32,
                warmup=warmup,
            )
            valid_mode = "FULL"
        else:
            if valid_loader is None:
                valid_loss_out = float("nan")
                pearson = {
                    "weighted_pearson": float("nan"),
                    "t0": float("nan"),
                    "t1": float("nan"),
                }
                valid_mode = "N/A"
            else:
                valid_loss_out, pearson = evaluate_fast_valid(
                    model_nn,
                    valid_loader=valid_loader,
                    device=device,
                    main_loss_fn=main_loss_fn,
                    main_loss_kwargs=main_loss_kwargs,
                    warmup=warmup,
                )
                valid_mode = "FAST"

        elapsed = time.time() - start_time

        wp = float(pearson.get("weighted_pearson", float("nan")))
        t0_wp = float(pearson.get("t0", float("nan")))
        t1_wp = float(pearson.get("t1", float("nan")))

        tqdm.write(
            f"Epoch: {epoch+1:03d} | Time: {elapsed:.2f}s | "
            f"Train Loss: {train_loss_out:.6f} | "
            f"Valid({valid_mode}) Loss: {valid_loss_out:.6f} | "
            f"WP: {wp:.6f} (t0={t0_wp:.6f}, t1={t1_wp:.6f})"
        )

        # ---- SAVE ----
        if save_each_epoch:
            pth_path = os.path.join(save_dir_pth, f"epoch_{epoch+1:03d}.pth")
            torch.save(model_nn.state_dict(), pth_path)

    return model_nn

In [41]:
def train_model(
    model_nn,
    train_loader,
    optimizer,
    num_epochs,
    device,
    X_valid, Y_valid, M_valid,

    main_loss_fn,
    main_loss_kwargs=None,

    aux_loss_fn=None,
    aux_loss_kwargs=None,

    aux_horizons=(1, 2, 3),
    aux_dim_per_horizon=4,
    aux_loss_weight=0.05,
    aux_channel_weights=None,
    warmup=100,

    valid_loader=None,
    use_full_valid=True,
    full_valid_every=0,

    save_dir_pth="models_pth",
    save_each_epoch=True,
):
    if main_loss_kwargs is None:
        main_loss_kwargs = {}
    if aux_loss_kwargs is None:
        aux_loss_kwargs = {}

    os.makedirs(save_dir_pth, exist_ok=True)

    model_nn = model_nn.to(device)

    aux_dim_total = len(aux_horizons) * aux_dim_per_horizon

    if aux_channel_weights is None:
        aux_channel_weights = torch.ones(aux_dim_total, dtype=torch.float32)
    aux_channel_weights = aux_channel_weights.to(device)
    aux_channel_weights = aux_channel_weights / aux_channel_weights.sum().clamp_min(1e-12)

    for epoch in range(num_epochs):
        start_time = time.time()

        model_nn.train()
        train_loss_sum = 0.0
        train_iters = 0

        with tqdm(
            total=len(train_loader),
            desc=f"Epoch {epoch+1} - Train",
            unit="batch",
            leave=False,
            dynamic_ncols=True,
        ) as pbar:
            for xb, yb, mb, y_aux_b, m_aux_b in train_loader:
                xb = xb.to(device, non_blocking=True)           # (B,T,F)
                yb = yb.to(device, non_blocking=True)           # (B,T,2)
                mb = mb.to(device, non_blocking=True)           # (B,T)
                y_aux_b = y_aux_b.to(device, non_blocking=True) # (B,T,D_total)
                m_aux_b = m_aux_b.to(device, non_blocking=True) # (B,T,D_total)

                pred_main, pred_aux_dict = model_nn(xb)         # main: (B,T,2)

                # ---- MAIN LOSS ----
                main_mask = build_main_loss_mask(mb, warmup=warmup)
                loss_main = main_loss_fn(pred_main, yb, main_mask, **main_loss_kwargs)

                # ---- AUX LOSS: собираем все горизонты в один tensor (B,T,D_total) ----
                pred_aux_all = torch.cat(
                    [pred_aux_dict[h] for h in aux_horizons],
                    dim=-1
                )  # (B,T,D_total)

                # loss по каждому aux-каналу отдельно: (D_total,)
                loss_aux_per_channel = aux_loss_fn(
                    pred_aux_all,
                    y_aux_b,
                    m_aux_b,
                    **aux_loss_kwargs,
                )

                # агрегируем по весам каналов
                loss_aux = (loss_aux_per_channel * aux_channel_weights).sum()

                loss = loss_main + aux_loss_weight * loss_aux

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model_nn.parameters(), max_norm=1.0)
                optimizer.step()

                train_loss_sum += float(loss.detach().item())
                train_iters += 1
                pbar.update(1)

        train_loss_out = train_loss_sum / max(train_iters, 1)

        # ---- VALID ----
        do_full = False
        if use_full_valid:
            if full_valid_every == 0:
                do_full = True
            else:
                do_full = (epoch % (full_valid_every + 1) == 0)

        if do_full:
            valid_loss_out, pearson = evaluate_full_valid(
                model_nn,
                X_valid=X_valid,
                Y_valid=Y_valid,
                M_valid=M_valid,
                device=device,
                main_loss_fn=main_loss_fn,
                main_loss_kwargs=main_loss_kwargs,
                batch_size=32,
                warmup=warmup,
            )
            valid_mode = "FULL"
        else:
            if valid_loader is None:
                valid_loss_out = float("nan")
                pearson = {
                    "weighted_pearson": float("nan"),
                    "t0": float("nan"),
                    "t1": float("nan"),
                }
                valid_mode = "N/A"
            else:
                valid_loss_out, pearson = evaluate_fast_valid(
                    model_nn,
                    valid_loader=valid_loader,
                    device=device,
                    main_loss_fn=main_loss_fn,
                    main_loss_kwargs=main_loss_kwargs,
                    warmup=warmup,
                )
                valid_mode = "FAST"

        elapsed = time.time() - start_time

        wp = float(pearson.get("weighted_pearson", float("nan")))
        t0_wp = float(pearson.get("t0", float("nan")))
        t1_wp = float(pearson.get("t1", float("nan")))

        tqdm.write(
            f"Epoch: {epoch+1:03d} | Time: {elapsed:.2f}s | "
            f"Train Loss: {train_loss_out:.6f} | "
            f"Valid({valid_mode}) Loss: {valid_loss_out:.6f} | "
            f"WP: {wp:.6f} (t0={t0_wp:.6f}, t1={t1_wp:.6f})"
        )

        if save_each_epoch:
            pth_path = os.path.join(save_dir_pth, f"epoch_{epoch+1:03d}.pth")
            torch.save(model_nn.state_dict(), pth_path)

    return model_nn

In [42]:
# main loss
main_loss_fn = masked_weighted_wpc_loss_seq2seq
main_loss_kwargs = {
    "eps": 1e-8,
    "clip": 6.0,
    "t0_weight": 0.8,
    "t1_weight": 0.2,
}

# aux loss
aux_loss_fn = masked_weighted_huber_loss_seq2seq
aux_loss_kwargs = {
    "delta": 1.0,
    "eps": 1e-3,
}

aux_loss_fn = masked_weighted_huber_loss_per_channel_seq2seq
aux_loss_kwargs = {
    "delta": 1.0,
    "eps": 1e-3,
}

AUX_DIM_TOTAL = Y_aux_all.shape[-1]
AUX_DIM_PER_HORIZON = AUX_DIM_TOTAL // len(AUX_HORIZONS)
#AUX_CHANNEL_WEIGHTS = torch.ones(AUX_DIM_TOTAL, dtype=torch.float32)
AUX_CHANNEL_WEIGHTS = torch.tensor([
    1.0, 1.0, 1.0, 1.0,   # h=1
    0.8, 0.8, 0.8, 0.8,   # h=2
    0.6, 0.6, 0.6, 0.6,   # h=3
], dtype=torch.float32)

print("AUX_DIM_TOTAL:", AUX_DIM_TOTAL)
print("AUX_DIM_PER_HORIZON:", AUX_DIM_PER_HORIZON)
print("AUX_CHANNEL_WEIGHTS:", AUX_CHANNEL_WEIGHTS)

assert AUX_DIM_TOTAL % len(AUX_HORIZONS) == 0

# aux_loss_weight=0.3 best

AUX_DIM_TOTAL: 12
AUX_DIM_PER_HORIZON: 4
AUX_CHANNEL_WEIGHTS: tensor([1.0000, 1.0000, 1.0000, 1.0000, 0.8000, 0.8000, 0.8000, 0.8000, 0.6000,
        0.6000, 0.6000, 0.6000])


### fold 1

In [43]:
fold_id = 1
tr_idx, va_idx = fold_splits[fold_id - 1]

train_pack, valid_pack = slice_fold_data(
    X_all, Y_all, M_all, Y_aux_all, M_aux_all, tr_idx, va_idx
)

train_loader, valid_loader = make_fullseq_loaders(
    train_pack,
    valid_pack,
    batch_size=32,
    seed=42 + fold_id,
)

model = GRUSeq2SeqMultiTaskRegressor(
    input_dim=len(FEATURE_COLS),
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_horizons=AUX_HORIZONS,
    hidden_dim=128,
    num_layers=3,
    dropout=0.1,
    clip_out=None,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4,
)

model_trained = train_model(
    model_nn=model,
    train_loader=train_loader,
    optimizer=optimizer,
    num_epochs=160,
    device=device,

    X_valid=valid_pack["X"],
    Y_valid=valid_pack["Y"],
    M_valid=valid_pack["M"],

    main_loss_fn=main_loss_fn,
    main_loss_kwargs=main_loss_kwargs,

    aux_loss_fn=aux_loss_fn,
    aux_loss_kwargs=aux_loss_kwargs,

    aux_horizons=AUX_HORIZONS,
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_loss_weight=0.3,
    aux_channel_weights=AUX_CHANNEL_WEIGHTS,
    warmup=100,

    valid_loader=valid_loader,
    use_full_valid=True,
    full_valid_every=0,

    save_dir_pth=f"models_fold_{fold_id}",
    save_each_epoch=False,
)

Epoch: 001 | Time: 5.71s | Train Loss: 1.100194 | Valid(FULL) Loss: 0.779489 | WP: 0.174286 (t0=0.253624, t1=0.094949)


Epoch: 002 | Time: 5.24s | Train Loss: 0.933206 | Valid(FULL) Loss: 0.733102 | WP: 0.209314 (t0=0.306300, t1=0.112328)


Epoch: 003 | Time: 5.15s | Train Loss: 0.896243 | Valid(FULL) Loss: 0.712230 | WP: 0.223292 (t0=0.330409, t1=0.116175)


Epoch: 004 | Time: 5.18s | Train Loss: 0.878364 | Valid(FULL) Loss: 0.700952 | WP: 0.233282 (t0=0.341252, t1=0.125311)


Epoch: 005 | Time: 5.15s | Train Loss: 0.867737 | Valid(FULL) Loss: 0.696418 | WP: 0.237832 (t0=0.347061, t1=0.128604)


...

Epoch: 156 | Time: 5.15s | Train Loss: 0.698481 | Valid(FULL) Loss: 0.657192 | WP: 0.214196 (t0=0.380483, t1=0.047909)


Epoch: 157 | Time: 5.20s | Train Loss: 0.699612 | Valid(FULL) Loss: 0.671155 | WP: 0.186921 (t0=0.353232, t1=0.020610)


Epoch: 158 | Time: 5.17s | Train Loss: 0.698269 | Valid(FULL) Loss: 0.662793 | WP: 0.199606 (t0=0.366361, t1=0.032851)


Epoch: 159 | Time: 5.17s | Train Loss: 0.699228 | Valid(FULL) Loss: 0.662216 | WP: 0.211120 (t0=0.371160, t1=0.051081)


Epoch: 160 | Time: 5.16s | Train Loss: 0.699701 | Valid(FULL) Loss: 0.654513 | WP: 0.225400 (t0=0.387531, t1=0.063269)


## fold 2

In [44]:
fold_id = 2
tr_idx, va_idx = fold_splits[fold_id - 1]

train_pack, valid_pack = slice_fold_data(
    X_all, Y_all, M_all, Y_aux_all, M_aux_all, tr_idx, va_idx
)

train_loader, valid_loader = make_fullseq_loaders(
    train_pack,
    valid_pack,
    batch_size=32,
    seed=42 + fold_id,
)

model = GRUSeq2SeqMultiTaskRegressor(
    input_dim=len(FEATURE_COLS),
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_horizons=AUX_HORIZONS,
    hidden_dim=128,
    num_layers=3,
    dropout=0.1,
    clip_out=None,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4,
)

model_trained = train_model(
    model_nn=model,
    train_loader=train_loader,
    optimizer=optimizer,
    num_epochs=160,
    device=device,

    X_valid=valid_pack["X"],
    Y_valid=valid_pack["Y"],
    M_valid=valid_pack["M"],

    main_loss_fn=main_loss_fn,
    main_loss_kwargs=main_loss_kwargs,

    aux_loss_fn=aux_loss_fn,
    aux_loss_kwargs=aux_loss_kwargs,

    aux_horizons=AUX_HORIZONS,
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_loss_weight=0.3,
    aux_channel_weights=AUX_CHANNEL_WEIGHTS,
    warmup=100,

    valid_loader=valid_loader,
    use_full_valid=True,
    full_valid_every=0,

    save_dir_pth=f"models_fold_{fold_id}",
    save_each_epoch=False,
)

Epoch: 001 | Time: 5.23s | Train Loss: 1.109725 | Valid(FULL) Loss: 0.776229 | WP: 0.172851 (t0=0.249323, t1=0.096379)


Epoch: 002 | Time: 5.23s | Train Loss: 0.929148 | Valid(FULL) Loss: 0.733777 | WP: 0.210882 (t0=0.297461, t1=0.124302)


Epoch: 003 | Time: 5.11s | Train Loss: 0.891703 | Valid(FULL) Loss: 0.710723 | WP: 0.217803 (t0=0.323150, t1=0.112456)


Epoch: 004 | Time: 5.13s | Train Loss: 0.871862 | Valid(FULL) Loss: 0.700339 | WP: 0.231816 (t0=0.338064, t1=0.125567)


Epoch: 005 | Time: 5.11s | Train Loss: 0.862824 | Valid(FULL) Loss: 0.694777 | WP: 0.236395 (t0=0.345241, t1=0.127549)


...

Epoch: 156 | Time: 5.13s | Train Loss: 0.700453 | Valid(FULL) Loss: 0.667888 | WP: 0.183741 (t0=0.371096, t1=-0.003614)


Epoch: 157 | Time: 5.16s | Train Loss: 0.696836 | Valid(FULL) Loss: 0.668738 | WP: 0.182518 (t0=0.381829, t1=-0.016793)


Epoch: 158 | Time: 5.11s | Train Loss: 0.698200 | Valid(FULL) Loss: 0.672663 | WP: 0.190678 (t0=0.369314, t1=0.012042)


Epoch: 159 | Time: 5.15s | Train Loss: 0.694412 | Valid(FULL) Loss: 0.668332 | WP: 0.182980 (t0=0.365791, t1=0.000170)


Epoch: 160 | Time: 5.12s | Train Loss: 0.699193 | Valid(FULL) Loss: 0.662454 | WP: 0.194707 (t0=0.388116, t1=0.001298)


### fold 3

In [45]:
fold_id = 3
tr_idx, va_idx = fold_splits[fold_id - 1]

train_pack, valid_pack = slice_fold_data(
    X_all, Y_all, M_all, Y_aux_all, M_aux_all, tr_idx, va_idx
)

train_loader, valid_loader = make_fullseq_loaders(
    train_pack,
    valid_pack,
    batch_size=32,
    seed=42 + fold_id,
)

model = GRUSeq2SeqMultiTaskRegressor(
    input_dim=len(FEATURE_COLS),
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_horizons=AUX_HORIZONS,
    hidden_dim=128,
    num_layers=3,
    dropout=0.1,
    clip_out=None,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4,
)

model_trained = train_model(
    model_nn=model,
    train_loader=train_loader,
    optimizer=optimizer,
    num_epochs=160,
    device=device,

    X_valid=valid_pack["X"],
    Y_valid=valid_pack["Y"],
    M_valid=valid_pack["M"],

    main_loss_fn=main_loss_fn,
    main_loss_kwargs=main_loss_kwargs,

    aux_loss_fn=aux_loss_fn,
    aux_loss_kwargs=aux_loss_kwargs,

    aux_horizons=AUX_HORIZONS,
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_loss_weight=0.3,
    aux_channel_weights=AUX_CHANNEL_WEIGHTS,
    warmup=100,

    valid_loader=valid_loader,
    use_full_valid=True,
    full_valid_every=0,

    save_dir_pth=f"models_fold_{fold_id}",
    save_each_epoch=False,
)

Epoch: 001 | Time: 5.25s | Train Loss: 1.091971 | Valid(FULL) Loss: 0.778549 | WP: 0.172125 (t0=0.250757, t1=0.093492)


Epoch: 002 | Time: 5.20s | Train Loss: 0.928812 | Valid(FULL) Loss: 0.731976 | WP: 0.206120 (t0=0.294919, t1=0.117321)


Epoch: 003 | Time: 5.09s | Train Loss: 0.890875 | Valid(FULL) Loss: 0.711190 | WP: 0.225595 (t0=0.322374, t1=0.128815)


Epoch: 004 | Time: 5.11s | Train Loss: 0.874409 | Valid(FULL) Loss: 0.702899 | WP: 0.232409 (t0=0.338015, t1=0.126802)


Epoch: 005 | Time: 5.11s | Train Loss: 0.864687 | Valid(FULL) Loss: 0.697258 | WP: 0.232279 (t0=0.345552, t1=0.119006)


...

Epoch: 156 | Time: 5.11s | Train Loss: 0.699457 | Valid(FULL) Loss: 0.677010 | WP: 0.208168 (t0=0.368731, t1=0.047605)


Epoch: 157 | Time: 5.13s | Train Loss: 0.699664 | Valid(FULL) Loss: 0.675730 | WP: 0.202137 (t0=0.362706, t1=0.041569)


Epoch: 158 | Time: 5.16s | Train Loss: 0.698835 | Valid(FULL) Loss: 0.692577 | WP: 0.176762 (t0=0.335376, t1=0.018148)


Epoch: 159 | Time: 5.09s | Train Loss: 0.696504 | Valid(FULL) Loss: 0.695170 | WP: 0.174549 (t0=0.340789, t1=0.008310)


Epoch: 160 | Time: 5.09s | Train Loss: 0.695122 | Valid(FULL) Loss: 0.680230 | WP: 0.187449 (t0=0.351652, t1=0.023246)


### fold 4

In [46]:
fold_id = 4
tr_idx, va_idx = fold_splits[fold_id - 1]

train_pack, valid_pack = slice_fold_data(
    X_all, Y_all, M_all, Y_aux_all, M_aux_all, tr_idx, va_idx
)

train_loader, valid_loader = make_fullseq_loaders(
    train_pack,
    valid_pack,
    batch_size=32,
    seed=42 + fold_id,
)

model = GRUSeq2SeqMultiTaskRegressor(
    input_dim=len(FEATURE_COLS),
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_horizons=AUX_HORIZONS,
    hidden_dim=128,
    num_layers=3,
    dropout=0.1,
    clip_out=None,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4,
)

model_trained = train_model(
    model_nn=model,
    train_loader=train_loader,
    optimizer=optimizer,
    num_epochs=160,
    device=device,

    X_valid=valid_pack["X"],
    Y_valid=valid_pack["Y"],
    M_valid=valid_pack["M"],

    main_loss_fn=main_loss_fn,
    main_loss_kwargs=main_loss_kwargs,

    aux_loss_fn=aux_loss_fn,
    aux_loss_kwargs=aux_loss_kwargs,

    aux_horizons=AUX_HORIZONS,
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_loss_weight=0.3,
    aux_channel_weights=AUX_CHANNEL_WEIGHTS,
    warmup=100,

    valid_loader=valid_loader,
    use_full_valid=True,
    full_valid_every=0,

    save_dir_pth=f"models_fold_{fold_id}",
    save_each_epoch=False,
)

Epoch: 001 | Time: 5.32s | Train Loss: 1.104061 | Valid(FULL) Loss: 0.772460 | WP: 0.167616 (t0=0.263264, t1=0.071968)


Epoch: 002 | Time: 5.20s | Train Loss: 0.927403 | Valid(FULL) Loss: 0.727818 | WP: 0.210850 (t0=0.315947, t1=0.105754)


Epoch: 003 | Time: 5.12s | Train Loss: 0.889879 | Valid(FULL) Loss: 0.711308 | WP: 0.227647 (t0=0.333916, t1=0.121378)


Epoch: 004 | Time: 5.10s | Train Loss: 0.872798 | Valid(FULL) Loss: 0.701974 | WP: 0.235483 (t0=0.345724, t1=0.125241)


Epoch: 005 | Time: 5.08s | Train Loss: 0.863981 | Valid(FULL) Loss: 0.696515 | WP: 0.239574 (t0=0.352345, t1=0.126802)


...

Epoch: 156 | Time: 5.12s | Train Loss: 0.700314 | Valid(FULL) Loss: 0.664997 | WP: 0.256186 (t0=0.383659, t1=0.128712)


Epoch: 157 | Time: 5.10s | Train Loss: 0.699307 | Valid(FULL) Loss: 0.671885 | WP: 0.240851 (t0=0.368747, t1=0.112954)


Epoch: 158 | Time: 5.14s | Train Loss: 0.696719 | Valid(FULL) Loss: 0.666305 | WP: 0.252420 (t0=0.380296, t1=0.124544)


Epoch: 159 | Time: 5.09s | Train Loss: 0.696314 | Valid(FULL) Loss: 0.672364 | WP: 0.248030 (t0=0.373066, t1=0.122995)


Epoch: 160 | Time: 5.11s | Train Loss: 0.701434 | Valid(FULL) Loss: 0.678435 | WP: 0.237915 (t0=0.362985, t1=0.112845)


### fold 5

In [47]:
fold_id = 5
tr_idx, va_idx = fold_splits[fold_id - 1]

train_pack, valid_pack = slice_fold_data(
    X_all, Y_all, M_all, Y_aux_all, M_aux_all, tr_idx, va_idx
)

train_loader, valid_loader = make_fullseq_loaders(
    train_pack,
    valid_pack,
    batch_size=32,
    seed=42 + fold_id,
)

model = GRUSeq2SeqMultiTaskRegressor(
    input_dim=len(FEATURE_COLS),
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_horizons=AUX_HORIZONS,
    hidden_dim=128,
    num_layers=3,
    dropout=0.1,
    clip_out=None,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4,
)

model_trained = train_model(
    model_nn=model,
    train_loader=train_loader,
    optimizer=optimizer,
    num_epochs=160,
    device=device,

    X_valid=valid_pack["X"],
    Y_valid=valid_pack["Y"],
    M_valid=valid_pack["M"],

    main_loss_fn=main_loss_fn,
    main_loss_kwargs=main_loss_kwargs,

    aux_loss_fn=aux_loss_fn,
    aux_loss_kwargs=aux_loss_kwargs,

    aux_horizons=AUX_HORIZONS,
    aux_dim_per_horizon=AUX_DIM_PER_HORIZON,
    aux_loss_weight=0.3,
    aux_channel_weights=AUX_CHANNEL_WEIGHTS,
    warmup=100,

    valid_loader=valid_loader,
    use_full_valid=True,
    full_valid_every=0,

    save_dir_pth=f"models_fold_{fold_id}",
    save_each_epoch=False,
)

Epoch: 001 | Time: 5.23s | Train Loss: 1.105342 | Valid(FULL) Loss: 0.780523 | WP: 0.163191 (t0=0.252719, t1=0.073662)


Epoch: 002 | Time: 5.22s | Train Loss: 0.931575 | Valid(FULL) Loss: 0.731812 | WP: 0.205946 (t0=0.309321, t1=0.102571)


Epoch: 003 | Time: 5.12s | Train Loss: 0.892851 | Valid(FULL) Loss: 0.711970 | WP: 0.222546 (t0=0.333814, t1=0.111278)


Epoch: 004 | Time: 5.12s | Train Loss: 0.875654 | Valid(FULL) Loss: 0.702086 | WP: 0.230844 (t0=0.345594, t1=0.116093)


Epoch: 005 | Time: 5.14s | Train Loss: 0.864993 | Valid(FULL) Loss: 0.698276 | WP: 0.232920 (t0=0.350760, t1=0.115080)


...

Epoch: 156 | Time: 5.05s | Train Loss: 0.693636 | Valid(FULL) Loss: 0.667513 | WP: 0.192008 (t0=0.387273, t1=-0.003258)


Epoch: 157 | Time: 5.04s | Train Loss: 0.693716 | Valid(FULL) Loss: 0.669457 | WP: 0.194928 (t0=0.386070, t1=0.003786)


Epoch: 158 | Time: 5.04s | Train Loss: 0.697772 | Valid(FULL) Loss: 0.675219 | WP: 0.199784 (t0=0.380080, t1=0.019489)


Epoch: 159 | Time: 5.06s | Train Loss: 0.690274 | Valid(FULL) Loss: 0.668354 | WP: 0.203429 (t0=0.385330, t1=0.021528)


Epoch: 160 | Time: 5.03s | Train Loss: 0.687552 | Valid(FULL) Loss: 0.676946 | WP: 0.187166 (t0=0.373224, t1=0.001108)
